# 피처 셀렉션 결과 괜찮은지 확인용도 (smote 비율은 1로만 두고 실행함)

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np

from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\test데이터\M19_도매_소매업_test.parquet'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.5


# ============================================================
# 2. 피처 파일
# ============================================================
feature_files = {
    "top25": r"C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\13번.피처셀렉션\M19_도매_소매업\lasso_features_top25.csv",
    "top30": r"C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\13번.피처셀렉션\M19_도매_소매업\lasso_features_top30.csv",
    "top35": r"C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\13번.피처셀렉션\M19_도매_소매업\lasso_features_top35.csv",
}


# ============================================================
# 3. 모델 정의
# ============================================================

models = {
    "LogisticRegression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs",
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbose=-1
    ),
    # "SVM": SVC(
    #     kernel="rbf", C=1.0, gamma="scale",
    #     probability=True, class_weight="balanced",
    #     random_state=RANDOM_STATE
    # )
}


# ============================================================
# 4. 데이터 로드
# ============================================================

train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

y_train = train[TARGET_COL]
y_test  = test[TARGET_COL]


# ============================================================
# 5. 모든 변수 합집합 생성
# ============================================================

all_features = set()

for feature_name, feature_path in feature_files.items():
    feature_df = pd.read_csv(feature_path)
    all_features.update(feature_df["column"].tolist())  # ← "column"으로 고정

all_features = [f for f in all_features if f in train.columns]

print("="*70)
print(f"전체 변수 개수 : {len(all_features)}")
print("="*70)


# ============================================================
# 6. 오버샘플링 함수 정의
# ============================================================

def apply_none(X_train, y_train, ratio=None):
    df_res = X_train.copy()
    df_res[TARGET_COL] = y_train.values
    return df_res


def apply_borderline_smote(X_train, y_train, ratio):
    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_res, y_res = smote.fit_resample(X_train, y_train)
    df_res = pd.DataFrame(X_res, columns=X_train.columns)
    df_res[TARGET_COL] = y_res
    return df_res


def apply_ctgan(X_train, y_train, ratio):
    n_majority = (y_train == 0).sum()
    n_minority = (y_train == 1).sum()
    n_target   = int(n_majority * ratio)
    n_to_gen   = max(n_target - n_minority, 0)

    if n_to_gen == 0:
        df_res = X_train.copy()
        df_res[TARGET_COL] = y_train.values
        return df_res

    minority_df = X_train[y_train == 1].copy()
    minority_df[TARGET_COL] = 1

    ctgan = CTGAN(epochs=100, verbose=False)
    ctgan.fit(minority_df, discrete_columns=[TARGET_COL])

    synthetic = ctgan.sample(n_to_gen)
    synthetic[TARGET_COL] = 1

    original_df = X_train.copy()
    original_df[TARGET_COL] = y_train.values

    df_res = pd.concat([original_df, synthetic], ignore_index=True)
    return df_res


# ============================================================
# 7. 실험 루프
# ============================================================

method_configs = {
    "None"            : (apply_none,             [None]),
    "BorderlineSMOTE" : (apply_borderline_smote, [1.0]),
    "CTGAN"           : (apply_ctgan,            [1.0])
}

results = []

for method_name, (oversample_fn, ratios) in method_configs.items():

    print("\n")
    print("="*70)
    print(f"오버샘플링 방식 : {method_name}")
    print("="*70)

    for ratio in ratios:

        print(f"\n  ratio = {ratio}")

        X_full          = train[all_features]
        train_resampled = oversample_fn(X_full, y_train, ratio)

        n0 = (train_resampled[TARGET_COL] == 0).sum()
        n1 = (train_resampled[TARGET_COL] == 1).sum()
        print(f"  데이터 구성 → 0: {n0}, 1: {n1}")

        for feature_name, feature_path in feature_files.items():

            feature_df   = pd.read_csv(feature_path)          # ← csv로 읽기
            use_features = [
                f for f in feature_df["column"].tolist()       # ← "column"으로 고정
                if f in train_resampled.columns
            ]

            X_train_final = train_resampled[use_features]
            y_train_final = train_resampled[TARGET_COL]
            X_test_final  = test[use_features]

            for model_name, model in models.items():

                print(f"    [{feature_name}] {model_name} 학습 중...")

                model.fit(X_train_final, y_train_final)

                y_prob = model.predict_proba(X_test_final)[:, 1]
                y_pred = (y_prob >= THRESHOLD).astype(int)

                results.append({
                    "Method"      : method_name,
                    "SMOTE_Ratio" : ratio if ratio is not None else "-",
                    "Feature_Set" : feature_name,
                    "Model"       : model_name,
                    "Num_Features": len(use_features),
                    "Accuracy"    : accuracy_score(y_test, y_pred),
                    "ROC_AUC"     : roc_auc_score(y_test, y_prob),
                    "PR_AUC"      : average_precision_score(y_test, y_prob),
                    "F1"          : f1_score(y_test, y_pred),
                    "Precision"   : precision_score(y_test, y_pred),
                    "Recall"      : recall_score(y_test, y_pred)
                })


# ============================================================
# 8. 결과 출력
# ============================================================

result_df = pd.DataFrame(results).round(4)

print("\n")
print("="*100)
print("전체 실험 결과")
print("="*100)
print(result_df.to_string(index=False))

for metric in ["PR_AUC", "Recall", "F1", "ROC_AUC"]:
    print(f"\n[{metric} 상위 5개]")
    print(
        result_df[["Method", "SMOTE_Ratio", "Feature_Set", "Model", metric]]
        .sort_values(metric, ascending=False)
        .head(5)
        .to_string(index=False)
    )

print("\n")
print("="*70)
print("오버샘플링 방식별 평균 성능 비교")
print("="*70)
print(
    result_df.groupby("Method")[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)

print("\n")
print("="*70)
print("방식 × 모델별 평균 성능")
print("="*70)
print(
    result_df.groupby(["Method", "Model"])[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)


# ============================================================
# 9. 저장
# ============================================================

result_df.to_csv(
    "Oversample_Method_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료 : Oversample_Method_Comparison.csv")



오버샘플링 방식 : None

  ratio = None
  데이터 구성 → 0: 27056, 1: 1055
    [top25] LogisticRegression 학습 중...
    [top25] RandomForest 학습 중...
    [top25] XGBoost 학습 중...
    [top25] LightGBM 학습 중...
    [top30] LogisticRegression 학습 중...
    [top30] RandomForest 학습 중...
    [top30] XGBoost 학습 중...
    [top30] LightGBM 학습 중...
    [top35] LogisticRegression 학습 중...
    [top35] RandomForest 학습 중...
    [top35] XGBoost 학습 중...
    [top35] LightGBM 학습 중...


오버샘플링 방식 : BorderlineSMOTE

  ratio = 1.0
  데이터 구성 → 0: 27056, 1: 27056
    [top25] LogisticRegression 학습 중...
    [top25] RandomForest 학습 중...
    [top25] XGBoost 학습 중...
    [top25] LightGBM 학습 중...
    [top30] LogisticRegression 학습 중...
    [top30] RandomForest 학습 중...
    [top30] XGBoost 학습 중...
    [top30] LightGBM 학습 중...
    [top35] LogisticRegression 학습 중...
    [top35] RandomForest 학습 중...
    [top35] XGBoost 학습 중...
    [top35] LightGBM 학습 중...


오버샘플링 방식 : CTGAN

  ratio = 1.0
  데이터 구성 → 0: 27056, 1: 27056
    [top25] LogisticRegre

# 피처셀렉션 4가지 결과에 따라서,  클래스 불균형 방식에 따라 달라지는 성능평가 결과 비교

## 머신러닝 - Logistic, RandomForest, XGB, LGBM, SVM

In [2]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np

from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH  = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\test데이터\M19_도매_소매업_test.parquet'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.5


# ============================================================
# 2. 피처 파일
# ============================================================
feature_files = {
    "top25": r"C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\13번.피처셀렉션\M19_도매_소매업\lasso_features_top26.csv"
}


# ============================================================
# 3. 모델 정의
# ============================================================

models = {
    "LogisticRegression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs",
        max_iter=1000, random_state=RANDOM_STATE
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbose=-1
    ),
}


# ============================================================
# 4. 데이터 로드
# ============================================================

train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

y_train = train[TARGET_COL]
y_test  = test[TARGET_COL]


# ============================================================
# 5. 모든 변수 합집합 생성
# ============================================================

all_features = set()

for feature_name, feature_path in feature_files.items():
    feature_df = pd.read_csv(feature_path)
    all_features.update(feature_df["column"].tolist())

all_features = [f for f in all_features if f in train.columns]

print("="*70)
print(f"전체 변수 개수 : {len(all_features)}")
print("="*70)


# ============================================================
# 6. 오버샘플링 함수 정의
# ============================================================

def apply_none(X_train, y_train, ratio=None):
    df_res = X_train.copy()
    df_res[TARGET_COL] = y_train.values
    return df_res


def apply_borderline_smote(X_train, y_train, ratio):
    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_res, y_res = smote.fit_resample(X_train, y_train)
    df_res = pd.DataFrame(X_res, columns=X_train.columns)
    df_res[TARGET_COL] = y_res
    return df_res


def apply_ctgan(X_train, y_train, ratio):
    n_majority = (y_train == 0).sum()
    n_minority = (y_train == 1).sum()
    n_target   = int(n_majority * ratio)
    n_to_gen   = max(n_target - n_minority, 0)

    if n_to_gen == 0:
        df_res = X_train.copy()
        df_res[TARGET_COL] = y_train.values
        return df_res

    minority_df = X_train[y_train == 1].copy()
    minority_df[TARGET_COL] = 1

    ctgan = CTGAN(epochs=100, verbose=False)
    ctgan.fit(minority_df, discrete_columns=[TARGET_COL])

    synthetic = ctgan.sample(n_to_gen)
    synthetic[TARGET_COL] = 1

    original_df = X_train.copy()
    original_df[TARGET_COL] = y_train.values

    df_res = pd.concat([original_df, synthetic], ignore_index=True)
    return df_res


# ============================================================
# 7. 실험 루프
# ============================================================

method_configs = {
    "None"           : (apply_none,              [None]),
    "BorderlineSMOTE": (apply_borderline_smote,  [0.2, 0.3, 0.5, 0.7, 1.0]),
}

results = []

for method_name, (oversample_fn, ratios) in method_configs.items():

    print("\n")
    print("="*70)
    print(f"오버샘플링 방식 : {method_name}")
    print("="*70)

    for ratio in ratios:

        print(f"\n  ratio = {ratio}")

        X_full          = train[all_features]
        train_resampled = oversample_fn(X_full, y_train, ratio)

        n0 = (train_resampled[TARGET_COL] == 0).sum()
        n1 = (train_resampled[TARGET_COL] == 1).sum()
        print(f"  데이터 구성 → 0: {n0}, 1: {n1}")

        for feature_name, feature_path in feature_files.items():

            feature_df   = pd.read_csv(feature_path)
            use_features = [
                f for f in feature_df["column"].tolist()
                if f in train_resampled.columns
            ]

            X_train_final = train_resampled[use_features]
            y_train_final = train_resampled[TARGET_COL]
            X_test_final  = test[use_features]

            # 오버샘플링 전 원본 train (train 성능 측정용)
            X_train_orig = train[use_features]

            for model_name, model in models.items():

                print(f"    [{feature_name}] {model_name} 학습 중...")

                model.fit(X_train_final, y_train_final)

                # Test 성능
                y_prob_test = model.predict_proba(X_test_final)[:, 1]
                y_pred_test = (y_prob_test >= THRESHOLD).astype(int)

                # Train 성능 (오버샘플링 전 원본 기준)
                y_prob_train = model.predict_proba(X_train_orig)[:, 1]
                y_pred_train = (y_prob_train >= THRESHOLD).astype(int)

                results.append({
                    "Method"          : method_name,
                    "SMOTE_Ratio"     : ratio if ratio is not None else "-",
                    "Feature_Set"     : feature_name,
                    "Model"           : model_name,
                    "Num_Features"    : len(use_features),
                    # Test
                    "Test_Accuracy"   : accuracy_score(y_test, y_pred_test),
                    "Test_ROC_AUC"    : roc_auc_score(y_test, y_prob_test),
                    "Test_PR_AUC"     : average_precision_score(y_test, y_prob_test),
                    "Test_F1"         : f1_score(y_test, y_pred_test),
                    "Test_Precision"  : precision_score(y_test, y_pred_test),
                    "Test_Recall"     : recall_score(y_test, y_pred_test),
                    # Train (오버샘플링 전 원본 기준)
                    "Train_Accuracy"  : accuracy_score(y_train, y_pred_train),
                    "Train_ROC_AUC"   : roc_auc_score(y_train, y_prob_train),
                    "Train_PR_AUC"    : average_precision_score(y_train, y_prob_train),
                    "Train_F1"        : f1_score(y_train, y_pred_train),
                    "Train_Precision" : precision_score(y_train, y_pred_train),
                    "Train_Recall"    : recall_score(y_train, y_pred_train),
                })


# ============================================================
# 8. 결과 출력
# ============================================================

result_df = pd.DataFrame(results).round(4)

print("\n")
print("="*100)
print("전체 실험 결과")
print("="*100)
print(result_df.to_string(index=False))

for metric in ["Test_PR_AUC", "Test_Recall", "Test_F1", "Test_ROC_AUC"]:
    print(f"\n[{metric} 상위 5개]")
    print(
        result_df[["Method", "SMOTE_Ratio", "Feature_Set", "Model",
                   metric, metric.replace("Test_", "Train_")]]
        .sort_values(metric, ascending=False)
        .head(5)
        .to_string(index=False)
    )

print("\n")
print("="*70)
print("오버샘플링 방식별 평균 성능 비교")
print("="*70)
print(
    result_df.groupby("Method")[
        ["Train_ROC_AUC", "Test_ROC_AUC",
         "Train_PR_AUC",  "Test_PR_AUC",
         "Train_F1",      "Test_F1",
         "Train_Recall",  "Test_Recall"]
    ].mean().round(4)
)

print("\n")
print("="*70)
print("방식 × 모델별 평균 성능")
print("="*70)
print(
    result_df.groupby(["Method", "Model"])[
        ["Train_ROC_AUC", "Test_ROC_AUC",
         "Train_PR_AUC",  "Test_PR_AUC",
         "Train_F1",      "Test_F1",
         "Train_Recall",  "Test_Recall"]
    ].mean().round(4)
)


# ============================================================
# 9. 저장
# ============================================================

result_df.to_csv(
    "Oversample_Method_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료 : Oversample_Method_Comparison.csv")

전체 변수 개수 : 30


오버샘플링 방식 : None

  ratio = None
  데이터 구성 → 0: 27056, 1: 1055
    [top25] LogisticRegression 학습 중...
    [top25] RandomForest 학습 중...
    [top25] XGBoost 학습 중...
    [top25] LightGBM 학습 중...


오버샘플링 방식 : BorderlineSMOTE

  ratio = 0.2
  데이터 구성 → 0: 27056, 1: 5411
    [top25] LogisticRegression 학습 중...
    [top25] RandomForest 학습 중...
    [top25] XGBoost 학습 중...
    [top25] LightGBM 학습 중...

  ratio = 0.3
  데이터 구성 → 0: 27056, 1: 8116
    [top25] LogisticRegression 학습 중...
    [top25] RandomForest 학습 중...
    [top25] XGBoost 학습 중...
    [top25] LightGBM 학습 중...

  ratio = 0.5
  데이터 구성 → 0: 27056, 1: 13528
    [top25] LogisticRegression 학습 중...
    [top25] RandomForest 학습 중...
    [top25] XGBoost 학습 중...
    [top25] LightGBM 학습 중...

  ratio = 0.7
  데이터 구성 → 0: 27056, 1: 18939
    [top25] LogisticRegression 학습 중...
    [top25] RandomForest 학습 중...
    [top25] XGBoost 학습 중...
    [top25] LightGBM 학습 중...

  ratio = 1.0
  데이터 구성 → 0: 27056, 1: 27056
    [top25] LogisticRegres

# 하이퍼파라미터 최적화 (Boderline SMOTE - 비율 0.30 )

In [4]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np
import optuna
import warnings

from imblearn.over_sampling import BorderlineSMOTE

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    f1_score,
    recall_score
)
from sklearn.model_selection import StratifiedKFold

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH   = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\train데이터\M19_도매_소매업_train.parquet'
TEST_PATH    = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\10,11,12번\test데이터\M19_도매_소매업_test.parquet'
FEATURE_PATH = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\13번.피처셀렉션\M19_도매_소매업\lasso_features_top26.csv'
OUTPUT_PATH  = r'C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\optuna_results.csv'

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
SMOTE_RATIO  = 0.3  
N_TRIALS     = 30
CV_FOLDS     = 3
THRESHOLD    = 0.5

# F1과 Recall 가중치 (합이 1이 되도록)
W_F1     = 0.4
W_RECALL = 0.6


# ============================================================
# 2. 데이터 로드
# ============================================================

train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

feature_df   = pd.read_csv(FEATURE_PATH)
use_features = [f for f in feature_df["column"].tolist() if f in train.columns]

X_train_raw = train[use_features]
y_train     = train[TARGET_COL]
X_test      = test[use_features]
y_test      = test[TARGET_COL]

print(f"피처 수    : {len(use_features)}")
print(f"Train shape: {X_train_raw.shape}")
print(f"Test  shape: {X_test.shape}")
print(f"클래스 분포 (train) → 0: {(y_train==0).sum():,}  1: {(y_train==1).sum():,}")


# ============================================================
# 3. SMOTE 적용
# ============================================================

smote = BorderlineSMOTE(
    sampling_strategy=SMOTE_RATIO,
    random_state=RANDOM_STATE,
    kind="borderline-1"
)
X_resampled, y_resampled = smote.fit_resample(X_train_raw, y_train)
print(f"\nSMOTE 후   → 0: {(y_resampled==0).sum():,}  1: {(y_resampled==1).sum():,}")

X_res_arr = X_resampled.values
y_res_arr = y_resampled.values


# ============================================================
# 4. CV 기반 목적함수 (F1 × W_F1 + Recall × W_RECALL)
# ============================================================

def cv_score(model, X, y):
    skf    = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    f1s, recalls = [], []
    for tr_idx, val_idx in skf.split(X, y):
        X_tr, X_val = X[tr_idx], X[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model.fit(X_tr, y_tr)
        y_prob = model.predict_proba(X_val)[:, 1]
        y_pred = (y_prob >= THRESHOLD).astype(int)
        f1s.append(f1_score(y_val, y_pred, zero_division=0))
        recalls.append(recall_score(y_val, y_pred, zero_division=0))

    mean_f1     = np.mean(f1s)
    mean_recall = np.mean(recalls)
    return W_F1 * mean_f1 + W_RECALL * mean_recall


# ============================================================
# 5. 모델별 Objective 함수 정의
# ============================================================

def objective_lr(trial):
    params = {
        "C"           : trial.suggest_float("C", 1e-3, 10.0, log=True),
        "penalty"     : trial.suggest_categorical("penalty", ["l1", "l2"]),
        "solver"      : "liblinear",
        "max_iter"    : 1000,
        "random_state": RANDOM_STATE,
    }
    return cv_score(LogisticRegression(**params), X_res_arr, y_res_arr)


def objective_rf(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 100, 500),
        "max_depth"       : trial.suggest_int("max_depth", 3, 10),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features"    : trial.suggest_categorical("max_features", ["sqrt", "log2"]),
        "random_state"    : RANDOM_STATE,
        "n_jobs"          : -1,
    }
    return cv_score(RandomForestClassifier(**params), X_res_arr, y_res_arr)


def objective_xgb(trial):
    params = {
        "n_estimators"     : trial.suggest_int("n_estimators", 100, 500),
        "learning_rate"    : trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth"        : trial.suggest_int("max_depth", 3, 8),
        "subsample"        : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree" : trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma"            : trial.suggest_float("gamma", 0.0, 5.0),
        "reg_alpha"        : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"       : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "use_label_encoder": False,
        "eval_metric"      : "aucpr",
        "random_state"     : RANDOM_STATE,
        "verbosity"        : 0,
    }
    return cv_score(XGBClassifier(**params), X_res_arr, y_res_arr)


def objective_lgbm(trial):
    params = {
        "n_estimators"    : trial.suggest_int("n_estimators", 100, 500),
        "learning_rate"   : trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "max_depth"       : trial.suggest_int("max_depth", 3, 8),
        "num_leaves"      : trial.suggest_int("num_leaves", 20, 150),
        "subsample"       : trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha"       : trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda"      : trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "random_state"    : RANDOM_STATE,
        "verbose"         : -1,
    }
    return cv_score(LGBMClassifier(**params), X_res_arr, y_res_arr)


# ============================================================
# 6. Optuna 실행
# ============================================================

model_objectives = {
    "LogisticRegression": (objective_lr,   LogisticRegression),
    "RandomForest"      : (objective_rf,   RandomForestClassifier),
    "XGBoost"           : (objective_xgb,  XGBClassifier),
    "LightGBM"          : (objective_lgbm, LGBMClassifier),
}

results = []

for model_name, (objective_fn, model_cls) in model_objectives.items():

    print(f"\n{'='*60}")
    print(f"▶ {model_name} 튜닝 중... (n_trials={N_TRIALS})")
    print(f"  최적화 기준: F1×{W_F1} + Recall×{W_RECALL}")
    print(f"{'='*60}")

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    study.optimize(objective_fn, n_trials=N_TRIALS, show_progress_bar=True)

    best_params = study.best_params
    best_cv     = study.best_value
    print(f"  최적 CV 스코어: {best_cv:.4f}")
    print(f"  최적 파라미터: {best_params}")

    # ── 최적 파라미터로 전체 SMOTE 데이터 재학습
    if model_name == "LogisticRegression":
        best_params.update({"solver": "liblinear", "max_iter": 1000, "random_state": RANDOM_STATE})
        final_model = LogisticRegression(**best_params)

    elif model_name == "RandomForest":
        best_params.update({"class_weight": "balanced", "random_state": RANDOM_STATE, "n_jobs": -1})
        final_model = RandomForestClassifier(**best_params)

    elif model_name == "XGBoost":
        best_params.update({"use_label_encoder": False, "eval_metric": "aucpr",
                            "random_state": RANDOM_STATE, "verbosity": 0})
        final_model = XGBClassifier(**best_params)

    elif model_name == "LightGBM":
        best_params.update({"random_state": RANDOM_STATE, "verbose": -1})
        final_model = LGBMClassifier(**best_params)

    final_model.fit(X_resampled, y_resampled)

    # Test 성능
    y_prob_test  = final_model.predict_proba(X_test)[:, 1]
    y_pred_test  = (y_prob_test >= THRESHOLD).astype(int)

    # Train 성능 (오버샘플링 전 원본 기준)
    y_prob_train = final_model.predict_proba(X_train_raw)[:, 1]
    y_pred_train = (y_prob_train >= THRESHOLD).astype(int)

    results.append({
        "Model"          : model_name,
        "Best_CV_Score"  : round(best_cv, 4),
        "Best_Params"    : str(best_params),
        # Test
        "Test_ROC_AUC"   : round(roc_auc_score(y_test, y_prob_test), 4),
        "Test_PR_AUC"    : round(average_precision_score(y_test, y_prob_test), 4),
        "Test_F1"        : round(f1_score(y_test, y_pred_test, zero_division=0), 4),
        "Test_Recall"    : round(recall_score(y_test, y_pred_test, zero_division=0), 4),
        # Train
        "Train_ROC_AUC"  : round(roc_auc_score(y_train, y_prob_train), 4),
        "Train_PR_AUC"   : round(average_precision_score(y_train, y_prob_train), 4),
        "Train_F1"       : round(f1_score(y_train, y_pred_train, zero_division=0), 4),
        "Train_Recall"   : round(recall_score(y_train, y_pred_train, zero_division=0), 4),
    })

    print(f"  Test  → ROC_AUC: {results[-1]['Test_ROC_AUC']}  F1: {results[-1]['Test_F1']}  Recall: {results[-1]['Test_Recall']}")
    print(f"  Train → ROC_AUC: {results[-1]['Train_ROC_AUC']}  F1: {results[-1]['Train_F1']}  Recall: {results[-1]['Train_Recall']}")


# ============================================================
# 7. 결과 출력 & 저장
# ============================================================

result_df = pd.DataFrame(results)

print("\n")
print("="*80)
print("📊 Optuna 튜닝 최종 결과")
print("="*80)
print(result_df[["Model", "Best_CV_Score",
                 "Test_ROC_AUC", "Test_F1", "Test_Recall",
                 "Train_ROC_AUC", "Train_F1", "Train_Recall"]].to_string(index=False))

result_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")

피처 수    : 30
Train shape: (28111, 30)
Test  shape: (11797, 30)
클래스 분포 (train) → 0: 27,056  1: 1,055

SMOTE 후   → 0: 27,056  1: 8,116

▶ LogisticRegression 튜닝 중... (n_trials=30)
  최적화 기준: F1×0.4 + Recall×0.6


Best trial: 17. Best value: 0.676599: 100%|██████████| 30/30 [01:02<00:00,  2.09s/it]


  최적 CV 스코어: 0.6766
  최적 파라미터: {'C': 9.706169662323752, 'penalty': 'l1'}
  Test  → ROC_AUC: 0.8929  F1: 0.3539  Recall: 0.5881
  Train → ROC_AUC: 0.924  F1: 0.4109  Recall: 0.5943

▶ RandomForest 튜닝 중... (n_trials=30)
  최적화 기준: F1×0.4 + Recall×0.6


Best trial: 24. Best value: 0.893366: 100%|██████████| 30/30 [05:55<00:00, 11.86s/it]


  최적 CV 스코어: 0.8934
  최적 파라미터: {'n_estimators': 267, 'max_depth': 10, 'min_samples_leaf': 14, 'max_features': 'sqrt'}
  Test  → ROC_AUC: 0.9394  F1: 0.3546  Recall: 0.6806
  Train → ROC_AUC: 0.9781  F1: 0.4783  Recall: 0.9346

▶ XGBoost 튜닝 중... (n_trials=30)
  최적화 기준: F1×0.4 + Recall×0.6


Best trial: 22. Best value: 0.969668: 100%|██████████| 30/30 [01:45<00:00,  3.53s/it]


  최적 CV 스코어: 0.9697
  최적 파라미터: {'n_estimators': 199, 'learning_rate': 0.28915905943817266, 'max_depth': 7, 'subsample': 0.8506840348032374, 'colsample_bytree': 0.6158393455662204, 'gamma': 0.03215021765739964, 'reg_alpha': 0.13309033500728948, 'reg_lambda': 0.014121238711257016}
  Test  → ROC_AUC: 0.9341  F1: 0.4265  Recall: 0.6167
  Train → ROC_AUC: 1.0  F1: 1.0  Recall: 1.0

▶ LightGBM 튜닝 중... (n_trials=30)
  최적화 기준: F1×0.4 + Recall×0.6


Best trial: 28. Best value: 0.976749: 100%|██████████| 30/30 [01:41<00:00,  3.37s/it]


  최적 CV 스코어: 0.9767
  최적 파라미터: {'n_estimators': 282, 'learning_rate': 0.12724238804262247, 'max_depth': 8, 'num_leaves': 148, 'subsample': 0.7242287309450507, 'colsample_bytree': 0.5831921077757534, 'reg_alpha': 0.06037830097118084, 'reg_lambda': 0.02150061887198424}
  Test  → ROC_AUC: 0.9127  F1: 0.4251  Recall: 0.5903
  Train → ROC_AUC: 1.0  F1: 1.0  Recall: 1.0


📊 Optuna 튜닝 최종 결과
             Model  Best_CV_Score  Test_ROC_AUC  Test_F1  Test_Recall  Train_ROC_AUC  Train_F1  Train_Recall
LogisticRegression         0.6766        0.8929   0.3539       0.5881         0.9240    0.4109        0.5943
      RandomForest         0.8934        0.9394   0.3546       0.6806         0.9781    0.4783        0.9346
           XGBoost         0.9697        0.9341   0.4265       0.6167         1.0000    1.0000        1.0000
          LightGBM         0.9767        0.9127   0.4251       0.5903         1.0000    1.0000        1.0000

저장 완료: C:\유비온프로젝트2\corporate-bankruptcy\데이터 전처리\optuna_results.csv


SMOTE 후 데이터가 많다면 먼저 ratio=0.3 한 가지 조건으로 SVM 단독 테스트해보고 속도 확인한 다음 전체 돌리는 걸 추천해.

In [5]:
# ============================================================
# 7. 결과 출력 & 저장
# ============================================================

result_df = pd.DataFrame(results)

print("\n")
print("="*80)
print("📊 Optuna 튜닝 최종 결과")
print("="*80)
print(result_df[["Model", "Best_CV_Score",
                 "Test_ROC_AUC", "Test_F1", "Test_Recall",
                 "Train_ROC_AUC", "Train_F1", "Train_Recall"]].to_string(index=False))

# ── Best Params 모델별 출력
print("\n")
print("="*80)
print("📋 모델별 최적 하이퍼파라미터")
print("="*80)
for row in results:
    print(f"\n[{row['Model']}]")
    import ast
    params = ast.literal_eval(row['Best_Params'])
    for k, v in params.items():
        print(f"  {k:<25} : {v}")

result_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print(f"\n저장 완료: {OUTPUT_PATH}")



📊 Optuna 튜닝 최종 결과
             Model  Best_CV_Score  Test_ROC_AUC  Test_F1  Test_Recall  Train_ROC_AUC  Train_F1  Train_Recall
LogisticRegression         0.6766        0.8929   0.3539       0.5881         0.9240    0.4109        0.5943
      RandomForest         0.8934        0.9394   0.3546       0.6806         0.9781    0.4783        0.9346
           XGBoost         0.9697        0.9341   0.4265       0.6167         1.0000    1.0000        1.0000
          LightGBM         0.9767        0.9127   0.4251       0.5903         1.0000    1.0000        1.0000


📋 모델별 최적 하이퍼파라미터

[LogisticRegression]
  C                         : 9.706169662323752
  penalty                   : l1
  solver                    : liblinear
  max_iter                  : 1000
  random_state              : 42

[RandomForest]
  n_estimators              : 267
  max_depth                 : 10
  min_samples_leaf          : 14
  max_features              : sqrt
  class_weight              : balanced
  random_state 

### LSTM
train / test DataFrame에 사업자등록번호, 회계년도 컬럼이 있어야 시퀀스 생성 가능

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd

from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)


# ============================================================
# LSTM 모델 정의
# ============================================================

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()


# ============================================================
# 시계열 시퀀스 생성 함수
# ============================================================

def make_sequences(df, feature_cols, target_col,
                   time_col="회계년도", id_col="사업자등록번호", window=3):
    X_list, y_list = [], []
    for _, group in df.groupby(id_col):
        group = group.sort_values(time_col).reset_index(drop=True)
        X = group[feature_cols].values
        y = group[target_col].values
        for i in range(len(group) - window + 1):
            X_list.append(X[i : i + window])
            y_list.append(y[i + window - 1])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


# ============================================================
# 오버샘플링 함수 정의
# ============================================================

def oversample_none(X_seq_train, y_seq_train, ratio=None):
    """클래스 불균형 처리 없음 - 원본 시퀀스 그대로 반환"""
    return X_seq_train, y_seq_train


def oversample_borderline_smote(X_seq_train, y_seq_train, ratio):
    """Flatten → BorderlineSMOTE → Reshape"""
    n_samples  = X_seq_train.shape[0]
    window     = X_seq_train.shape[1]
    n_features = X_seq_train.shape[2]

    X_flat = X_seq_train.reshape(n_samples, window * n_features)

    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_flat_res, y_res = smote.fit_resample(X_flat, y_seq_train)
    X_seq_res = X_flat_res.reshape(-1, window, n_features)

    return X_seq_res, y_res


def oversample_ctgan(X_seq_train, y_seq_train, ratio):
    """마지막 시점 단면 데이터로 CTGAN 학습 → 합성 → 시퀀스 복원"""
    window     = X_seq_train.shape[1]
    n_features = X_seq_train.shape[2]

    n_majority = int((y_seq_train == 0).sum())
    n_minority = int((y_seq_train == 1).sum())
    n_target   = int(n_majority * ratio)
    n_to_gen   = max(n_target - n_minority, 0)

    if n_to_gen == 0:
        return X_seq_train, y_seq_train

    minority_last = X_seq_train[y_seq_train == 1, -1, :]
    minority_df   = pd.DataFrame(minority_last)

    ctgan = CTGAN(epochs=100, verbose=False)
    ctgan.fit(minority_df, discrete_columns=[])
    synthetic_last = ctgan.sample(n_to_gen).values.astype(np.float32)

    minority_seqs = X_seq_train[y_seq_train == 1]
    mean_prefix   = minority_seqs.mean(axis=0)

    synthetic_seqs = np.tile(mean_prefix[np.newaxis, :, :], (n_to_gen, 1, 1))
    synthetic_seqs[:, -1, :] = synthetic_last

    X_seq_res = np.concatenate([X_seq_train, synthetic_seqs], axis=0)
    y_res      = np.concatenate([
        y_seq_train,
        np.ones(n_to_gen, dtype=np.float32)
    ])

    return X_seq_res, y_res


# ============================================================
# LSTM 학습 및 평가 공통 함수
# ============================================================

def train_and_evaluate_lstm(X_seq_res, y_res, X_seq_test, y_seq_test,
                             n_features, epochs=30, batch_size=64, lr=1e-3):

    X_tr = torch.tensor(X_seq_res,  dtype=torch.float32)
    y_tr = torch.tensor(y_res,       dtype=torch.float32)
    X_te = torch.tensor(X_seq_test,  dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True
    )

    model     = LSTMClassifier(input_size=n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # None 방식은 불균형 그대로 → pos_weight로 보완
    n_neg      = (y_res == 0).sum()
    n_pos      = (y_res == 1).sum()
    pos_weight = torch.tensor([n_neg / n_pos])
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).numpy()

    y_pred = (y_prob >= THRESHOLD).astype(int)

    return {
        "Accuracy"  : accuracy_score(y_seq_test, y_pred),
        "ROC_AUC"   : roc_auc_score(y_seq_test, y_prob),
        "PR_AUC"    : average_precision_score(y_seq_test, y_prob),
        "F1"        : f1_score(y_seq_test, y_pred),
        "Precision" : precision_score(y_seq_test, y_pred),
        "Recall"    : recall_score(y_seq_test, y_pred)
    }


# ============================================================
# LSTM 실험 루프
# 방식별 ratio 설정이 다르므로 method_configs로 관리
# 총 조합 : None(1) + SMOTE(3) + CTGAN(3) × 4 feature × 3 window = 84개
# ============================================================

WINDOW_SIZES = [2, 3, 5]

method_configs = {
    # 방식명             : (함수,                      ratio 리스트)
    "None"              : (oversample_none,            [None]),
    "BorderlineSMOTE"   : (oversample_borderline_smote,[0.2, 0.3, 0.5]),
    "CTGAN"             : (oversample_ctgan,           [0.2, 0.3, 0.5])
}

lstm_results = []

for method_name, (oversample_fn, ratios) in method_configs.items():

    print("\n")
    print("="*70)
    print(f"[LSTM] 오버샘플링 방식 : {method_name}")
    print("="*70)

    for feature_name, feature_path in feature_files.items():

        feature_df   = pd.read_csv(feature_path)
        feature_col  = (
            "final_feature" if "final_feature" in feature_df.columns else "feature"
        )
        use_features = [f for f in feature_df[feature_col].tolist() if f in train.columns]

        print(f"\n  Feature Set : {feature_name} ({len(use_features)}개)")

        for window in WINDOW_SIZES:

            X_seq_train, y_seq_train = make_sequences(
                train, use_features, TARGET_COL, window=window
            )
            X_seq_test, y_seq_test = make_sequences(
                test, use_features, TARGET_COL, window=window
            )

            if (y_seq_train == 1).sum() == 0 or (y_seq_test == 1).sum() == 0:
                print(f"  window={window} → 부실 샘플 없음, 스킵")
                continue

            for ratio in ratios:

                print(f"  window={window}, ratio={ratio} 오버샘플링 중...")

                X_seq_res, y_res = oversample_fn(X_seq_train, y_seq_train, ratio)

                print(f"    → 0: {int((y_res==0).sum())}, 1: {int((y_res==1).sum())}")

                metrics = train_and_evaluate_lstm(
                    X_seq_res, y_res,
                    X_seq_test, y_seq_test,
                    n_features=len(use_features)
                )

                lstm_results.append({
                    "Method"      : method_name,
                    "Feature_Set" : feature_name,
                    "Window"      : window,
                    "SMOTE_Ratio" : ratio if ratio is not None else "-",
                    "Num_Features": len(use_features),
                    **metrics
                })


# ============================================================
# 결과 출력 및 저장
# ============================================================

lstm_df = pd.DataFrame(lstm_results).round(4)

print("\n")
print("="*100)
print("LSTM 오버샘플링 방식 비교 결과")
print("="*100)
print(lstm_df.to_string(index=False))

# 방식별 평균 성능 비교 ← 핵심
print("\n")
print("="*70)
print("방식별 평균 성능 비교")
print("="*70)
print(
    lstm_df.groupby("Method")[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)

# 방식 × window별 평균
print("\n")
print("="*70)
print("방식 × Window별 평균 성능")
print("="*70)
print(
    lstm_df.groupby(["Method", "Window"])[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)

lstm_df.to_csv(
    "LSTM_Oversample_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료 : LSTM_Oversample_Comparison.csv")